In [ ]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 39.4 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import os, sqlite3, jwt, bcrypt, datetime, time, streamlit as st
import plotly.graph_objects as go
from streamlit_option_menu import option_menu
import smtplib
import secrets
import hmac
from email.message import EmailMessage
import re

# 🚀 Force Streamlit Light Theme
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#ffd803"\nbackgroundColor="#f9fcfc"\nsecondaryBackgroundColor="#e3f6f5"\ntextColor="#2d334a"\n')

st.set_page_config(page_title="Maritime Freight Portal", page_icon="⛵", layout="wide", initial_sidebar_state="expanded")

COLORS = {
    "bg_main": "#f9fcfc", "bg_sidebar": "#e3f6f5", "bg_card": "#ffffff", "bg_card_alt": "#bae8e8",
    "text_main": "#2d334a", "text_heading": "#272343", "text_muted": "#64748b",
    "accent": "#ffd803", "accent_hover": "#e6c300", "accent_text": "#272343",
    "border": "#272343", "border_light": "#bae8e8", "success": "#34d399", "danger": "#f87171"
}

JWT_SECRET = os.environ.get("JWT_SECRET")

if not JWT_SECRET:
    raise RuntimeError("JWT_SECRET is not configured.")

# 🚀 Neo-Brutalist CSS + Visible Sidebar Re-Open Button
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600&display=swap');
    html, body, .stApp {{ background: {COLORS['bg_main']} !important; font-family: 'Inter', sans-serif !important; color: {COLORS['text_main']} !important; }}

    /* 🚀 DO NOT HIDE HEADER! Only hide footer and bottom decoration so toggle arrow stays visible! */
    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}
    header {{ background: transparent !important; z-index: 999999 !important; }}

    /* 🚀 Highlight Streamlit's Native Sidebar Re-Open Button (Top-Left) as a Yellow Badge! */
    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important; display: flex !important; opacity: 1 !important;
        background-color: {COLORS['accent']} !important; border: 2px solid {COLORS['border']} !important;
        border-radius: 8px !important; padding: 6px !important; margin: 8px !important;
        box-shadow: 3px 3px 0px {COLORS['border']} !important;
    }}
    button[kind="header"] svg, div[data-testid="stSidebarCollapsedControl"] svg {{
        fill: {COLORS['text_heading']} !important; color: {COLORS['text_heading']} !important; stroke: {COLORS['text_heading']} !important;
    }}

    .block-container {{ padding: 2rem 2.5rem !important; max-width: 1200px; }}
    h1, h2, h3, h4 {{ font-family: 'Poppins', sans-serif !important; color: {COLORS['text_heading']} !important; }}
    label p {{ font-weight: 600 !important; color: {COLORS['text_heading']} !important; }}

    /* Inputs */
    div[data-baseweb="base-input"], div[data-baseweb="select"] > div {{ background-color: transparent !important; border: none !important; }}
    div[data-baseweb="input"], div[data-baseweb="select"] {{ background-color: {COLORS['bg_card']} !important; border: 2px solid {COLORS['border']} !important; border-radius: 10px !important; }}
    div[data-baseweb="input"]:focus-within {{ border-color: {COLORS['accent']} !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; }}
    input, div[data-baseweb="select"] span {{ color: {COLORS['text_main']} !important; -webkit-text-fill-color: {COLORS['text_main']} !important; }}

    /* Buttons */
    div[data-testid="stButton"] button {{
        background-color: {COLORS['accent']} !important; color: {COLORS['accent_text']} !important;
        border: 2px solid {COLORS['border']} !important; border-radius: 10px !important;
        font-family: 'Inter', sans-serif !important; font-weight: 700 !important; font-size: 14px !important;
        height: 48px !important; min-height: 48px !important; white-space: nowrap !important;
        display: flex !important; align-items: center !important; justify-content: center !important;
        padding: 0px 16px !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; width: 100%; transition: all 0.2s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{
        background-color: {COLORS['accent_hover']} !important; transform: translate(-2px, -2px) !important;
        box-shadow: 6px 6px 0px {COLORS['border']} !important;
    }}
    section[data-testid="stSidebar"] {{ background: {COLORS['bg_sidebar']} !important; border-right: 2px solid {COLORS['border']} !important; }}
    .pn-card {{ background: {COLORS['bg_card']}; border: 2px solid {COLORS['border']}; border-radius: 14px; padding: 24px; box-shadow: 4px 4px 0px {COLORS['border_light']}; }}
</style>
""", unsafe_allow_html=True)

def get_db():
    return sqlite3.connect(
        "/content/infosys_portal.db",
        check_same_thread=False
    )
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False


def valid_email(email):
    return bool(
        re.fullmatch(
            r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
            email.strip()
        )
    )


def valid_password(password):
    return (
        isinstance(password, str)
        and len(password) >= 8
        and not any(ch.isspace() for ch in password)
        and re.search(r"[A-Z]", password)
        and re.search(r"[a-z]", password)
        and re.search(r"[0-9]", password)
        and re.search(r"[^A-Za-z0-9\s]", password)
    )



def send_otp_email(receiver_email, otp):
    sender_email = os.environ["EMAIL_ADDRESS"]
    app_password = os.environ["EMAIL_PASSWORD"]

    message = EmailMessage()
    message["Subject"] = "Maritime Freight Portal - Password Reset OTP"
    message["From"] = sender_email
    message["To"] = receiver_email
    message.set_content(
        f"Your password reset OTP is {otp}.\n"
        "It expires in 5 minutes. Do not share this code."
    )

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(sender_email, app_password)
        server.send_message(message)


def generate_otp():
    return f"{secrets.randbelow(1_000_000):06d}"


def login_lock_remaining(email):
    """Return remaining login-lock seconds, or 0 when not locked."""
    with get_db() as c:
        row = c.execute(
            "SELECT attempts, last_attempt FROM login_attempts WHERE email=?",
            (email,)
        ).fetchone()
    if row and row[0] >= 3:
        remaining = 1200 - (time.time() - row[1])
        if remaining > 0:
            return int(remaining)
    return 0


def reset_lock_remaining(email):
    """Return remaining reset-lock seconds, or 0 when not locked."""
    with get_db() as c:
        row = c.execute(
            "SELECT locked_until FROM reset_lockouts WHERE email=?",
            (email,)
        ).fetchone()
    return max(0, int(row[0] - time.time())) if row else 0


def lock_reset_account(email):
    with get_db() as c:
        c.execute(
            "INSERT OR REPLACE INTO reset_lockouts (email, locked_until) VALUES (?, ?)",
            (email, time.time() + 600)
        )
        c.commit()


with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
        password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS reset_lockouts (
            email TEXT PRIMARY KEY,
            locked_until REAL NOT NULL
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS login_attempts (
            email TEXT PRIMARY KEY,
            attempts INTEGER NOT NULL DEFAULT 0,
            last_attempt REAL NOT NULL DEFAULT 0
        )
    """)
    if not conn.execute("SELECT id FROM users WHERE email='infosys@ai'").fetchone():
        conn.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                     ("Administrator", "infosys@ai", hash_txt("admin@123"), "What is your pet name?", hash_txt("admin")))

def make_jwt(email): return jwt.encode({"email": email, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)}, JWT_SECRET, algorithm="HS256")
def verify_jwt(token):
    if not token or not isinstance(token, str):
        return None

    try:
        return jwt.decode(
            token,
            JWT_SECRET,
            algorithms=["HS256"]
        )
    except jwt.InvalidTokenError:
        return None


for k, v in [
    ("token", None),
    ("page", "Login"),
    ("reset_email", None),
    ("reset_mode", None),
    ("otp_code", None),
    ("otp_expires", None),
    ("otp_verified", False),
    ("otp_attempts", 0)
]:
    if k not in st.session_state:
        st.session_state[k] = v


def navigate(p): st.session_state.page = p; st.rerun()

def auth_header(title, sub="Intelligent Analytics Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:40px;margin-bottom:10px;">⛵</div>
        <h1 style="font-size:2rem !important;margin:0;">Maritime Freight Portal</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;"><span style="font-size:1.1rem;font-weight:700;color:{COLORS['text_heading']};">{title}</span></div>
    """, unsafe_allow_html=True)

# ============================================================
# PAGE ROUTING
# ============================================================
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="••••••••")
            st.markdown("<br>", unsafe_allow_html=True)


            col_l, col_c, col_r = st.columns([1, 1.15, 1.3])

            if col_l.button("Sign In →", use_container_width=True):
                if not email or not pwd:
                    st.error("Please enter your email and password.")
                else:
                    with get_db() as c:
                        c.execute("""
                            CREATE TABLE IF NOT EXISTS login_attempts (
                                email TEXT PRIMARY KEY,
                                attempts INTEGER NOT NULL DEFAULT 0,
                                last_attempt REAL NOT NULL DEFAULT 0
                            )
                        """)

                        row = c.execute(
                            "SELECT attempts, last_attempt FROM login_attempts WHERE email=?",
                            (email,)
                        ).fetchone()

                        attempts, last_attempt = row if row else (0, 0)
                        now = time.time()

                        if attempts >= 3 and now - last_attempt < 1200:
                            remaining = int(1200 - (now - last_attempt))
                            minutes = (remaining + 59) // 60
                            st.error(
                                f"Account temporarily locked. Try again in {minutes} minute(s)."
                            )
                        else:
                            if attempts >= 3:
                                c.execute(
                                    "DELETE FROM login_attempts WHERE email=?",
                                    (email,)
                                )
                                attempts = 0

                            user = c.execute(
                                "SELECT password_hash FROM users WHERE email=?",
                                (email,)
                            ).fetchone()

                            if user and check_txt(pwd, user[0]):
                                c.execute(
                                    "DELETE FROM login_attempts WHERE email=?",
                                    (email,)
                                )
                                st.session_state.token = make_jwt(email)
                                navigate("Dashboard")
                            else:
                                attempts += 1
                                c.execute("""
                                    INSERT OR REPLACE INTO login_attempts
                                    (email, attempts, last_attempt)
                                    VALUES (?, ?, ?)
                                """, (email, attempts, now))

                                if attempts >= 3:
                                    st.error(
                                        "Account locked for 20 minutes after 3 failed login attempts."
                                    )
                                else:
                                    st.error(
                                        f"Invalid credentials. Attempt {attempts}/3."
                                    )

            if col_c.button("Create Account", use_container_width=True):
                navigate("Signup")

            if col_r.button("Forgot Password", use_container_width=True):
                navigate("Forgot")


        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Maritime Freight Portal today")
            uname = st.text_input("Full name / Username", placeholder="Jane Doe")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters")
            confirm_pwd = st.text_input("Confirm password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
            sa = st.text_input("Your answer", placeholder="Security answer")
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Create Account & Login →", use_container_width=True):
                if not all([uname.strip(), email.strip(), pwd, confirm_pwd, sa.strip()]):
                    st.error("Please fill in all required fields.")
                elif not valid_email(email):
                    st.error("Enter a valid email address.")
                elif not valid_password(pwd):
                    st.error(
                       "Password must be at least 8 characters, contain no spaces, "
                       "and include uppercase, lowercase, a number, and a special symbol."
                     )
                elif any(ch.isspace() for ch in sa):
                    st.error("Security answer must not contain spaces.")
                elif pwd != confirm_pwd:
                    st.error("Passwords do not match.")
                else:
                    try:
                        with get_db() as c:
                            c.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)", (uname, email, hash_txt(pwd), sq, hash_txt(sa)))
                        st.session_state.token = None
                        st.success("✅ Account created!")
                        time.sleep(1)
                        navigate("Login")
                    except sqlite3.IntegrityError:
                        st.error("❌ Email or Username already registered.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True): navigate("Login")


        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")

            # Step A: Select verification method
            if not st.session_state.reset_email:
                email = st.text_input(
                    "Registered email address",
                    placeholder="you@example.com"
                ).lower().strip()

                col_sq, col_otp = st.columns(2)

                if col_sq.button("Via Security Question", use_container_width=True):
                    if login_lock_remaining(email) > 0 or reset_lock_remaining(email) > 0:
                        remaining = max(login_lock_remaining(email), reset_lock_remaining(email))
                        st.error(f"Password recovery is blocked while the account is locked. Try again in about {max(1, (remaining + 59) // 60)} minute(s).")
                        st.stop()
                    with get_db() as c:
                        user = c.execute(
                            "SELECT security_question FROM users WHERE email=?",
                            (email,)
                        ).fetchone()

                    if user:
                        st.session_state.reset_email = email
                        st.session_state.reset_mode = "sq"
                        st.session_state.sq_p = user[0]
                        st.rerun()
                    else:
                        st.error("Email not found in the portal.")

                if col_otp.button("Via OTP", use_container_width=True):
                    if login_lock_remaining(email) > 0 or reset_lock_remaining(email) > 0:
                        remaining = max(login_lock_remaining(email), reset_lock_remaining(email))
                        st.error(f"Password recovery is blocked while the account is locked. Try again in about {max(1, (remaining + 59) // 60)} minute(s).")
                        st.stop()
                    with get_db() as c:
                        user = c.execute(
                            "SELECT email FROM users WHERE email=?",
                            (email,)
                        ).fetchone()

                    if not email or not user:
                        st.error("Enter a registered portal email address.")
                    else:
                        otp = generate_otp()

                        try:
                            send_otp_email(email, otp)

                            st.session_state.reset_email = email
                            st.session_state.reset_mode = "otp"
                            st.session_state.otp_code = otp
                            st.session_state.otp_expires = time.time() + 300
                            st.session_state.otp_verified = False
                            st.session_state.otp_attempts = 0

                            st.success("OTP sent successfully.")
                            st.rerun()

                        except Exception as e:
                            st.error("Unable to send OTP. Check your email configuration.")
                            print("OTP error:", e)


            # Step B: Reset using the security question
            elif st.session_state.get("reset_mode") == "sq":
                st.info(f"Security Question: {st.session_state.sq_p}")

                ans = st.text_input("Your answer")
                npw = st.text_input(
                    "New password (minimum 8 characters)",
                    type="password"
                )
                confirm_npw = st.text_input(
                    "Confirm new password",
                    type="password"
                )

                if st.button("Reset Password →", use_container_width=True):
                    if not ans:
                        st.error("Please enter your security answer.")
                    elif any(ch.isspace() for ch in ans):
                        st.error("Security answer must not contain spaces.")
                    elif not valid_password(npw):
                        st.error("Password must be at least 8 characters, contain no spaces, and include uppercase, lowercase, a number, and a special symbol.")
                    elif npw != confirm_npw:
                        st.error("Passwords do not match.")
                    else:
                        verified_email = st.session_state.reset_email

                        with get_db() as c:
                            user = c.execute(
                                """
                                SELECT security_answer_hash
                                FROM users
                                WHERE lower(email) = lower(?)
                                """,
                                (verified_email.strip(),)
                            ).fetchone()

                            if not user or not check_txt(ans, user[0]):
                                st.error("Incorrect security answer.")
                            else:
                                new_hash = hash_txt(npw)
                                result = c.execute(
                                    """
                                    UPDATE users
                                    SET password_hash = ?
                                    WHERE lower(email) = lower(?)
                                    """,
                                    (new_hash, verified_email.strip())
                                )
                                c.commit()

                                saved = c.execute(
                                    """
                                    SELECT password_hash
                                    FROM users
                                    WHERE lower(email) = lower(?)
                                    """,
                                    (verified_email.strip(),)
                                ).fetchone()

                                if result.rowcount == 1 and saved and check_txt(npw, saved[0]):
                                    c.execute(
                                        """
                                        DELETE FROM login_attempts
                                        WHERE lower(email) = lower(?)
                                        """,
                                        (verified_email.strip(),)
                                    )
                                    c.commit()

                                    st.session_state.token = None
                                    st.session_state.reset_email = None
                                    st.session_state.reset_mode = None
                                    st.session_state.page = "Login"

                                    st.success("Password updated successfully. Please log in.")
                                    st.rerun()
                                else:
                                    st.error("Password update could not be verified. Please try again.")

            # Step C: Verify OTP
            elif st.session_state.get("reset_mode") == "otp":
                st.info(f"OTP sent to {st.session_state.reset_email}.")

                if not st.session_state.otp_verified:
                     entered_otp = st.text_input("Enter the six-digit OTP", max_chars=6)

                     if st.button("Verify OTP", use_container_width=True):
                        if time.time() >= st.session_state.otp_expires:
                            st.session_state.otp_code = None
                            st.error("OTP expired. Cancel and request a new one.")
                        elif st.session_state.otp_attempts >= 2:
                            st.session_state.otp_code = None
                            st.error("OTP attempts exhausted. Request a new OTP.")
                        elif (
                            st.session_state.otp_code is not None
                            and hmac.compare_digest(
                                entered_otp,
                                st.session_state.otp_code
                            )
                        ):
                            st.session_state.otp_verified = True
                            st.session_state.otp_code = None
                            st.success("OTP verified successfully.")
                            st.rerun()
                        else:
                            st.session_state.otp_attempts += 1
                            if st.session_state.otp_attempts >= 2:
                                lock_reset_account(st.session_state.reset_email)
                                st.session_state.otp_code = None
                                st.session_state.reset_email = None
                                st.session_state.reset_mode = None
                                st.session_state.otp_verified = False
                                st.error("Two incorrect OTP attempts. Account recovery is locked for 10 minutes.")
                                st.rerun()
                            else:
                                st.error("Incorrect OTP. One attempt remaining.")

                else:
                    npw = st.text_input("New password (minimum 8 characters)", type="password")
                    confirm_npw = st.text_input("Confirm new password", type="password")

                    if st.button("Update Password →", use_container_width=True):
                        if not valid_password(npw):
                            st.error("Password must be at least 8 characters, contain no spaces, and include uppercase, lowercase, a number, and a special symbol.")
                        elif npw != confirm_npw:
                            st.error("Passwords do not match.")
                        elif time.time() >= st.session_state.otp_expires:
                            st.session_state.otp_verified = False
                            st.error("OTP verification expired. Request a new OTP.")
                        else:
                            verified_email = st.session_state.reset_email



                            with get_db() as c:
                                new_hash = hash_txt(npw)

                                result = c.execute(
                                    """
                                    UPDATE users
                                    SET password_hash = ?
                                    WHERE lower(email) = lower(?)
                                    """,
                                    (new_hash, verified_email.strip())
                                )

                                if result.rowcount == 1:
                                    c.commit()

                                    saved = c.execute(
                                        """
                                        SELECT password_hash FROM users
                                        WHERE lower(email) = lower(?)
                                        """,
                                        (verified_email.strip(),)
                                    ).fetchone()

                                    if saved and check_txt(npw, saved[0]):
                                        c.execute(
                                            """
                                            DELETE FROM login_attempts
                                            WHERE lower(email) = lower(?)
                                            """,
                                            (verified_email.strip(),)
                                        )
                                        c.execute("DELETE FROM reset_lockouts WHERE lower(email)=lower(?)", (verified_email.strip(),))
                                        c.commit()

                                        st.success(
                                            "Password updated and verified. Please log in."
                                        )

                                        st.session_state.reset_email = None
                                        st.session_state.reset_mode = None
                                        st.session_state.otp_code = None
                                        st.session_state.otp_expires = None
                                        st.session_state.otp_verified = False
                                        st.session_state.otp_attempts = 0
                                        st.rerun()
                                    else:
                                        st.error(
                                            "Password update could not be verified in the database."
                                        )
                                else:
                                    st.error(
                                        "No matching account was updated. "
                                        "Check the registered email."
                                    )



            # Cancel and return to login
            if st.button("← Cancel", use_container_width=True):
                for key, value in [
                    ("reset_email", None),
                    ("reset_mode", None),
                    ("otp_code", None),
                    ("otp_expires", None),
                    ("otp_verified", False),
                    ("otp_attempts", 0)
                ]:
                    st.session_state[key] = value

                st.session_state.page = "Login"
                st.rerun()


# ============================================================
# DASHBOARDS (ADMIN vs USER)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    email = payload["email"]
    with get_db() as c: uname = c.execute("SELECT username FROM users WHERE email=?", (email,)).fetchone()[0]

    # 🚀 SIDEBAR MENU
    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px;text-align:center;">
            <div style="font-size:28px;">⛵</div>
            <div style="font-weight:700;font-size:16px;color:{COLORS['text_heading']};">Maritime Freight Portal</div>
            <div style="font-size:11px;color:{COLORS['text_muted']};">{"Admin Panel" if email=="infosys@ai" else "Intelligent Analytics"}</div>
        </div><hr style="border-color:{COLORS['border_light']};">
        """, unsafe_allow_html=True)

        opts = ["Dashboard", "Settings", "Logout"] if email=="infosys@ai" else ["Dashboard", "Analytics", "Reports", "Logout"]
        menu = option_menu(None, opts, icons=["house", "gear", "box-arrow-right"] if email=="infosys@ai" else ["house", "graph-up", "file-text", "box-arrow-right"],
                           styles={"container": {"background-color": COLORS['bg_sidebar']}, "nav-link-selected": {"background-color": COLORS['accent'], "color": COLORS['accent_text']}})
        if menu == "Logout":
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

    # 🚀 ADMIN DASHBOARD (infosys@ai)
    if email == "infosys@ai":
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⛵ Maritime Freight Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Admin Control Panel</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">🛡️ {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown(f"""
        <div class="pn-card" style="text-align:center;padding:60px 20px;">
            <h1 style="font-size:36px !important;margin-bottom:10px;">🛡️ Admin Dashboard</h1>
            <p style="color:{COLORS['text_muted']};font-size:16px;font-weight:500;">Welcome to the Administrator area.</p>
        </div>
        """, unsafe_allow_html=True)


        st.markdown("## Registered Users")

        with get_db() as c:
            users = c.execute(
                """
                SELECT username, email
                FROM users
                WHERE email != ?
                ORDER BY username
                """,
                ("infosys@ai",)
            ).fetchall()

        if users:
            st.dataframe(
                [
                    {"Username": username, "Email": user_email}
                    for username, user_email in users
                ],
                use_container_width=True,
                hide_index=True
            )
        else:
            st.info("No registered users found.")


    # 🚀 REGULAR USER DASHBOARD
    else:
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⛵ Maritime Freight Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Analytics Dashboard</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">👤 {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [(c1, "📄", "Documents Indexed", "128"), (c2, "🔍", "Searches Today", "47"),
                                    (c3, "📊", "Efficiency Score", "98.4%"), (c4, "🛡️", "Security Status", "Secured")]:
            col.markdown(f"""
            <div class="pn-card" style="text-align:center;">
                <div style="font-size:28px;">{icon}</div>
                <div style="font-size:26px;font-weight:700;color:{COLORS['text_heading']};">{val}</div>
                <div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;">{lbl}</div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(go.Indicator(mode="gauge+number", value=92, title={"text": "System Health Index", "font": {"color": COLORS['text_heading'], "size": 14}},
                        gauge={"axis": {"range": [0, 100]}, "bar": {"color": COLORS['accent']}, "bgcolor": COLORS['bg_card_alt'], "borderwidth": 1, "bordercolor": COLORS['border']}))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": COLORS['text_main'], "family": "Inter"}, height=260, margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)


Writing app.py


In [ ]:

import os
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

# 1. Load credentials from Colab Secrets
NGROK_TOKEN = userdata.get("NGROKAUTHTOKEN")
os.environ["EMAIL_ADDRESS"] = userdata.get("EMAIL_ADDRESS")
os.environ["EMAIL_PASSWORD"] = userdata.get("EMAIL_PASSWORD")
os.environ["JWT_SECRET"] = userdata.get("JWT_SECRET")

# Validate required secrets without displaying their values
if not NGROK_TOKEN:
    raise ValueError("NGROKAUTHTOKEN is missing from Colab Secrets.")

for key in ("EMAIL_ADDRESS", "EMAIL_PASSWORD", "JWT_SECRET"):
    if not os.environ.get(key):
        raise ValueError(f"{key} is missing from Colab Secrets.")

# 2. Stop old tunnels and Streamlit processes
try:
    ngrok.kill()
except Exception:
    pass

subprocess.run(
    ["bash", "-lc", "pkill -f '[s]treamlit run app.py' || true"],
    check=False
)
time.sleep(2)

# 3. Configure ngrok only if its existing config is not already usable
# set_auth_token() does not accept save=False in your installed version.
try:
    tunnels = ngrok.get_tunnels()
    ngrok_configured = True
except Exception:
    tunnels = []
    ngrok_configured = False

if not ngrok_configured:
    # The existing ngrok config may be stale or missing.
    # Set the token once using the installed pyngrok API.
    ngrok.set_auth_token(NGROK_TOKEN)

# 4. Start Streamlit
process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True
)

# 5. Wait for Streamlit to start
time.sleep(5)

if process.poll() is not None:
    error_output = process.stderr.read() if process.stderr else ""
    raise RuntimeError(
        "Streamlit failed to start. Error details:\n" + error_output[-3000:]
    )

# 6. Open the public tunnel
try:
    tunnel = ngrok.connect(8501)
    print("=" * 55)
    print("Maritime Freight Portal is running:")
    print(tunnel.public_url)
    print("=" * 55)
    print("Keep this cell running while using the portal.")
except Exception:
    process.terminate()
    raise


Maritime Freight Portal is running:
https://makeover-theatrics-folic.ngrok-free.dev
Keep this cell running while using the portal.


In [ ]:
from pyngrok import ngrok

ngrok.kill()
print("Existing ngrok tunnels stopped.")

Existing ngrok tunnels stopped.
